# Ensemble Feature Importance & SHAP Analysis

This notebook provides an in-depth interpretation of the evolved ensemble model using **SHAP**. We explore both the ensemble's global behavior and the mechanics of the individual base learners.

**Prerequisites:**
- You must have run `post_hoc_evaluation.ipynb` to generate the `deployed_ensemble_model.joblib` file.
- Ensure `shap` is installed in your environment (`pip install shap`).

**Workflow:**
1. **Load Model & Data**: Reconstruct the environment from the latest experiment.
2. **Ensemble-wide SHAP**: Global feature importance for the aggregate model.
3. **Base Learner Performance**: Identify the strongest and weakest models in the committee.
4. **Base Learner SHAP**: Compare feature usage across different model types within the ensemble.
5. **Local Case Study**: See how a single prediction is composed from diverse base learner inputs.
6. **Advanced SHAP Visualizations**: Explore additional SHAP plots for deeper insights.

### Key Concept: The "Evolved Weight"
Our Genetic Algorithm (GA) assigns a weight to each base learner in the ensemble. This weight represents the model's contribution to the final decision. 
- **Trust Factor**: Higher weights are awarded to models that generalized well during evolution and capture unique signals.
- **Committee Logic**: The final prediction is a result of all learners 'voting'. A model with a weight of 0.6 has double the influence of one with 0.3.
- **Feature Interaction**: A model might have a high weight even if its individual AUC is low, provided it identifies patterns that other models miss.

In [ ]:
import ast
import glob
import json
import os
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import shap
from sklearn.metrics import roc_auc_score

from ml_grid.util.evaluate_ensemble_methods import EnsembleEvaluator
from ml_grid.util.global_params import global_parameters

# Suppress verbose sklearn and shap warnings
warnings.filterwarnings(
    "ignore", category=UserWarning, module="sklearn.utils.validation"
)
warnings.filterwarnings("ignore", category=FutureWarning)

# 1. Setup paths (Mirroring post_hoc_evaluation logic)
CONFIG_PATH = "config.yml"
BASE_PROJECT_DIR = "HFE_GA_experiments"
global_params = global_parameters(config_path=CONFIG_PATH)
INPUT_DATA_PATH = global_params.input_csv_path
OUTCOME_VAR = f"outcome_var_{global_params.outcome_var_n}"

# RAM Safety Caps for SHAP evaluations
MAX_EVALS_GLOBAL_HARD_CAP = 5000
MAX_EVALS_LEARNER_HARD_CAP = 2000

# SHAP Sampling Size
SHAP_SAMPLE_SIZE = 200

# Locate the latest run
list_of_runs = glob.glob(os.path.join(BASE_PROJECT_DIR, "*"))
if not list_of_runs:
    raise FileNotFoundError("No experiment runs found.")
latest_run_dir = max(list_of_runs, key=os.path.getctime)
model_path = os.path.join(latest_run_dir, "deployed_ensemble_model.joblib")

if not os.path.exists(model_path):
    raise FileNotFoundError(
        f"Model not found at {model_path}. Please run post_hoc_evaluation.ipynb first."
    )

print(f"Loading Model from: {model_path}")
my_ensemble = joblib.load(model_path)

# Attempt to load the exact configuration used for this model to ensure feature alignment
log_path = os.path.join(latest_run_dir, "final_grid_score_log.csv")
init_params = {"resample": None}
best_run = None
if os.path.exists(log_path):
    log_df = pd.read_csv(log_path)
    if not log_df.empty:
        best_run = log_df.sort_values("auc", ascending=False).iloc[0]
        init_params = best_run.to_dict()
        print(
            f"Reconstructing data using parameters from run log: corr={init_params.get('corr')}, resample={init_params.get('resample')}"
        )

# Instantiate EnsembleEvaluator to correctly load data and get original_feature_names
evaluator = EnsembleEvaluator(
    input_csv_path=INPUT_DATA_PATH,
    outcome_variable=OUTCOME_VAR,
    initial_param_dict=init_params,
)

# The definitive mapping used to decode ensemble bitmasks
full_feature_mapping = evaluator.original_feature_names
if best_run is not None and "original_feature_names" in best_run:
    try:
        full_feature_mapping = json.loads(best_run["original_feature_names"])
    except Exception:
        pass

if not full_feature_mapping:
    raise ValueError(
        "Could not determine feature mapping. Ensure EnsembleEvaluator or best_run['original_feature_names'] provides it."
    )

# Use the fallback original feature list saved in the ensemble, or full mapping
my_ensemble.feature_names = getattr(my_ensemble, "original_feature_names_used", full_feature_mapping)

# FIX: Align data against ALL available pool features so models find their missing columns!
X_train_aligned = evaluator.ml_grid_object.X_train.reindex(
    columns=my_ensemble.feature_names
).fillna(0)
y_train = evaluator.ml_grid_object.y_train
X_test = evaluator.ml_grid_object.X_test.reindex(
    columns=my_ensemble.feature_names
).fillna(0)
y_test = evaluator.ml_grid_object.y_test

print("--- Feature Alignment Summary ---")
print(f"Total features exposed to ensemble: {X_train_aligned.shape[1]}")

print("\n--- ALIGNMENT DIAGNOSTICS ---")
if hasattr(my_ensemble, "ensemble_arch") and my_ensemble.ensemble_arch:
    first_mask = my_ensemble.ensemble_arch[0][2]
    mask_len = len(first_mask)
    mapping_len = len(my_ensemble.feature_names)
    print(f"1. Chromosome Mask Format: {type(first_mask).__name__} of length {mask_len}")
    print(f"2. Name Mapping Count: {mapping_len}")

    if (
        isinstance(first_mask, (list, tuple, np.ndarray))
        and len(first_mask) > 0
        and isinstance(first_mask[0], str)
    ):
        print("-> Format detected: Mask contains explicit feature name strings.")
        active_features = [f for f in first_mask if f in my_ensemble.feature_names]
    elif not all(
        isinstance(x, (int, np.integer)) and x in [0, 1] for x in first_mask
    ):
        print("-> Format detected: Mask contains feature indices.")
        active_features = [
            my_ensemble.feature_names[i]
            for i in first_mask
            if 0 <= i < len(my_ensemble.feature_names)
        ]
    else:
        print("-> Format detected: Mask is a binary bitmask (0/1).")
        if mask_len == mapping_len:
            active_features = [
                my_ensemble.feature_names[i] for i, val in enumerate(first_mask) if val == 1
            ]
        else:
            print("!!! WARNING: Binary mask length mismatch. Applying adaptive truncation alignment.")
            active_features = []
            for i, val in enumerate(first_mask):
                if int(val) == 1 and i < len(my_ensemble.feature_names):
                    active_features.append(my_ensemble.feature_names[i])

    print(f"-> Successfully aligned {len(active_features)} features via matching fallback logic.")
else:
    print(
        "WARNING: Could not verify bitmask length as 'ensemble_arch' is not available or empty."
    )

print(f"3. X_train_aligned shape: {X_train_aligned.shape}")
print(f"4. X_test shape: {X_test.shape}")
print("-----------------------------\n")

print(f"Fitting ensemble on aligned X_train (shape: {X_train_aligned.shape})...")
my_ensemble.fit(X_train_aligned, y_train)

print("\nSklearn compatible ensemble is ready!")

In [ ]:
print("=== ENSEMBLE INTERNAL FEATURE VERIFICATION ===")
print(f"Total features passed to ensemble wrapper: {X_train_aligned.shape[1]}")
print(f"Number of successfully fitted base learners: {len(my_ensemble.fitted_models)}")
print("-" * 50)

# Inspect the features used by each individual base learner inside the ensemble
for idx, (model, active_features, weight) in enumerate(my_ensemble.fitted_models):
    model_name = type(model).__name__
    print(f"Base Learner #{idx + 1}: {model_name}")
    print(f"  -> Evolved Weight: {weight}")
    print(f"  -> Count of features it actually trained on: {len(active_features)}")
    print(f"  -> Explicit feature list: {active_features}")
    
    # Double-check Scikit-Learn's internal state if applicable
    if hasattr(model, "n_features_in_"):
        print(f"  -> Verified by sklearn underlying model (n_features_in_): {model.n_features_in_}")
    print("-" * 50)

In [ ]:
my_ensemble

In [ ]:
my_ensemble.fitted_models

In [ ]:
# Final verification: Ensure X_test now contains ALL features required by the ensemble
if (
    hasattr(my_ensemble, "all_req_features")
    and my_ensemble.all_req_features is not None
):
    missing_after_alignment = [
        f for f in my_ensemble.all_req_features if f not in X_test.columns
    ]
    if missing_after_alignment:
        raise ValueError(
            f"CRITICAL ERROR: X_test is still missing features after alignment: {missing_after_alignment}. This should not happen if reindex worked correctly."
        )
    else:
        print(
            "Final check: X_test successfully aligned with ensemble's required features."
        )

### Ensemble Anatomy
Before diving into SHAP, we visualize the structure of the ensemble. This section provides a detailed breakdown of the machine learning algorithms selected by the GA, their assigned weights, and the size of their unique feature subsets. This is crucial for reporting the model's composition and understanding the diversity of the committee.

In [ ]:
print("--- Global Ensemble Configuration ---")
weight_method = getattr(my_ensemble, "weight_method", "N/A")
print(f"Ensemble Weighting Strategy: {weight_method}")
print(f"Total Base Learners: {len(my_ensemble.fitted_models)}")

anatomy_data = []
for i, (model, features, weight) in enumerate(my_ensemble.fitted_models):
    # Extract model name, handling sklearn Pipeline wrappers if present
    model_name = type(model).__name__
    if model_name == "Pipeline":
        model_name = f"Pipeline({type(model.steps[-1][1]).__name__})"

    anatomy_data.append(
        {
            "Learner Index": i + 1,
            "Algorithm": model_name,
            "Weight": weight,
            "Feature Count": len(features),
            "Features (Sample)": ", ".join(features[:5])
            + ("..." if len(features) > 5 else ""),
        }
    )

anatomy_df = pd.DataFrame(anatomy_data)
display(anatomy_df)

# Visualize weights vs feature count
plt.figure(figsize=(10, 5))
sns.scatterplot(
    data=anatomy_df, x="Feature Count", y="Weight", hue="Algorithm", s=150, alpha=0.8
)
plt.title("Ensemble Anatomy: Model Complexity vs. Evolved Trust (Weight)")
plt.grid(True, linestyle="--", alpha=0.4)
plt.show()

### 2. Global Ensemble SHAP
This section treats the entire ensemble as a single unit. We want to know: **what features drive the final decision?** We use a permutation-based explainer which is safe for ensembles. It shows the 'big picture' of feature importance.

In [ ]:
import gc

import pandas as pd

# 1. More aggressive sampling for RAM safety
# 50 samples for the explanation is usually sufficient for global trends
# We sample a representative subset of the test data for SHAP explanation to manage computational cost and memory.
X_sample = X_test.sample(min(SHAP_SAMPLE_SIZE, len(X_test)), random_state=42)
# 2. Use K-Means to summarize background data.
# This is much more RAM-efficient than using raw samples as a masker.
# It represents the 'expected' value of features using 10 centroids, which serves as the background for SHAP calculations.
# We extract .data (numpy array) to ensure compatibility with modern SHAP maskers
X_bg_summary = shap.kmeans(X_test, 10).data
num_features_global = X_test.shape[1]
# Permutation explainer strictly requires at least 2*N + 1 evaluations.
required_max_evals_global = 2 * num_features_global + 500
if required_max_evals_global > MAX_EVALS_GLOBAL_HARD_CAP:
    print(
        f"WARNING: Required evals ({required_max_evals_global}) exceeds safety cap ({MAX_EVALS_GLOBAL_HARD_CAP})."
    )
    print(
        "Dynamically increasing to avoid ValueError from SHAP. This may increase memory and runtime."
    )
print(f"Calculating RAM-safe SHAP values for {num_features_global} features...")


# Wrap predict_proba to only return probability of the positive class (class 1)
# This wrapper ensures the explainer works with the ensemble's specific prediction output
def ensemble_proba_wrapper(x):
    # Use the subsetted features to match the input 'x' provided by the explainer
    df_x = pd.DataFrame(x, columns=X_test.columns)
    return my_ensemble.predict_proba(df_x)[:, 1]


# Verify model output variation
# A warning is issued if the model's output is nearly constant, as SHAP values would then be uninformative.
sample_preds = ensemble_proba_wrapper(X_sample.values)
print(
    f"Ensemble prediction range on sample: [{sample_preds.min():.4f}, {sample_preds.max():.4f}]"
)
if np.ptp(sample_preds) < 1e-5:
    print("WARNING: Model output is nearly constant. SHAP values will likely be zero.")
# 3. Clean memory before heavy lifting
gc.collect()
# Initialize the SHAP Explainer with the ensemble's prediction function and the K-Means summarized background.
# Using shap.maskers.Independent is suitable for model-agnostic explainers like KernelExplainer.
explainer = shap.Explainer(
    ensemble_proba_wrapper, masker=shap.maskers.Independent(X_bg_summary)
)
# 4. Execute explanation
# This computes the SHAP values for the sampled instances.
shap_values = explainer(
    X_sample, max_evals=required_max_evals_global, batch_size=50, silent=False
)
# 1. Global Bar Plot: Shows feature importance by average absolute SHAP value.
plt.figure(figsize=(10, 6))
shap.plots.bar(shap_values, max_display=min(20, num_features_global))
plt.show()
# 2. Summary (Bee-swarm): Shows distribution and directionality of impact.
# Red points = high feature values, Blue = low. Position on X-axis is impact on the probability score.
try:
    plt.figure(figsize=(10, 6))
    # Limit max_display to keep the plot readable and RAM-safe
    shap.plots.beeswarm(shap_values, max_display=min(20, num_features_global))
except Exception as e:
    print(f"Beeswarm plot failed (likely due to low variance): {e}")
# 3. Global Dependence (Scatter) Plots: See non-linear relationships and interactions
print("Generating Global Dependence Plots for top 2 features...")
try:
    # Identify top 2 features by mean absolute SHAP value
    top_2_indices = np.argsort(np.abs(shap_values.values).mean(0))[-2:][::-1]
    for idx in top_2_indices:
        feat_name = shap_values.feature_names[idx]
        plt.figure()
        shap.plots.scatter(shap_values[:, feat_name], color=shap_values)
        plt.show()
except Exception as e:
    print(f"Global dependence plot failed: {e}")
# 4. Heatmap plot: Provides a bird's-eye view of feature effects across samples.
# Each row is a sample, each column is a feature, and the color indicates the SHAP value.
# This helps identify patterns of feature influence across different predictions.
print("Generating Heatmap plot...")
try:
    plt.figure(figsize=(10, 6))
    # Limit max_display to keep the plot readable and RAM-safe, especially with many features.
    shap.plots.heatmap(shap_values, max_display=min(20, num_features_global))
    plt.show()
except Exception as e:
    print(f"Heatmap plot failed: {e}")

### 3. Individual Base Learner Performance
This section evaluates each base learner within the ensemble independently on its specific feature subset. This helps us understand the individual strengths and weaknesses of the models that comprise the ensemble, and how their evolved weights correlate with their performance.

In [ ]:
learner_stats = []
for i, (model, features, weight) in enumerate(my_ensemble.fitted_models):
    # features is already a list of strings (column names)
    X_subset = X_test[features]

    if X_subset.empty:
        print(f"Warning: Learner {i+1} has no features selected. Assigning AUC of 0.5.")
        probs = np.full(len(y_test), 0.5)
        auc = 0.5
    else:
        probs = model.predict_proba(X_subset)[:, 1]
        auc = roc_auc_score(y_test, probs)

    learner_stats.append(
        {
            "Learner_ID": i + 1,
            "Type": type(model).__name__,
            "Weight": weight,
            "Individual_AUC": auc,
            "Num_Features": len(features),
        }
    )

stats_df = pd.DataFrame(learner_stats).sort_values("Individual_AUC", ascending=False)
print("--- Base Learner Performance Comparison ---")
display(stats_df)

plt.figure(figsize=(10, 5))
sns.scatterplot(
    data=stats_df,
    x="Individual_AUC",
    y="Weight",
    size="Num_Features",
    hue="Type",
    alpha=0.7,
)
plt.title("Learner Quality vs. Evolved Weight")
plt.grid(True, linestyle="--", alpha=0.6)

# Add labels to identify specific learners on the plot
for i in range(stats_df.shape[0]):
    plt.text(
        stats_df.Individual_AUC.iloc[i] + 0.005,
        stats_df.Weight.iloc[i] + 0.005,
        str(int(stats_df.Learner_ID.iloc[i])),
        fontsize=9,
        weight="semibold",
    )
plt.tight_layout()
plt.show()

### 4. Deep Dive: Top Base Learner vs. Bottom Base Learner
We compare the feature importance of the best performing base learner versus the one with the lowest individual AUC to see if they utilize different data signals.

In [ ]:
def plot_learner_shap(learner_idx, title):
    model, features, weight = my_ensemble.fitted_models[learner_idx]

    # 1. Sample a smaller set of instances for explanation
    # A small sample is used for SHAP explanation to manage computational cost and memory.
    X_subset_explain = X_test[features].sample(
        min(SHAP_SAMPLE_SIZE, len(X_test)), random_state=42
    )

    if X_subset_explain.empty:
        print(
            f"Warning: Skipping SHAP plot for {title} as it has no features selected."
        )
        return

    num_features_learner = X_subset_explain.shape[1]
    # Permutation explainer strictly requires at least 2N + 1 evaluations.
    min_evals_learner = 2 * num_features_learner + 1
    required_max_evals_learner = max(1000, min_evals_learner)

    if required_max_evals_learner > MAX_EVALS_LEARNER_HARD_CAP:
        print(
            f"WARNING: Required evals ({required_max_evals_learner}) for {title} exceeds safety cap. Adjusting to avoid crash."
        )
    if num_features_learner <= 11:
        required_max_evals_learner = max(
            required_max_evals_learner, 2**num_features_learner
        )

    print(
        f"Calculating RAM-safe SHAP values for {num_features_learner} features for {title}..."
    )

    # 2. Use K-Means to summarize the background data for the explainer
    # This is crucial for RAM safety, especially with many features, providing a representative background.
    X_bg_summary_learner = shap.kmeans(X_test[features], 10).data

    # 3. Clean memory before heavy lifting
    gc.collect()

    # Wrap model.predict_proba to ensure it receives a DataFrame with feature names
    # This prevents the 'X does not have valid feature names' UserWarning from sklearn
    def learner_predict_wrapper(x):
        return model.predict_proba(pd.DataFrame(x, columns=features))

    # Initialize the SHAP Explainer for the individual base learner.
    # Using KernelExplainer/Explainer as a generic fallback, with the summarized background.
    learner_explainer = shap.Explainer(
        learner_predict_wrapper, masker=shap.maskers.Independent(X_bg_summary_learner)
    )
    l_shap_values = learner_explainer(
        X_subset_explain,
        max_evals=required_max_evals_learner,
        batch_size=50,
        silent=False,
    )

    # Handle multi-output (class 1)
    if len(l_shap_values.shape) == 3:
        vals = l_shap_values[:, :, 1]
    else:
        vals = l_shap_values

    # Summary plot (Bar/Beeswarm): Shows the overall feature importance for this specific base learner.
    plt.figure()
    plt.title(f"{title} ({type(model).__name__})")
    shap.summary_plot(
        vals, X_subset_explain, show=False, max_display=min(20, num_features_learner)
    )
    plt.show()

    # 4. Dependence Plots: See how feature values relate to impact (for top 2 features).
    # These plots show the relationship between a feature's value and its SHAP value,
    # often revealing non-linear relationships or interactions with other features.
    print(f"Generating scatter plot for top feature of {title}...")
    try:
        # Find top 2 features by mean absolute SHAP value to focus on the most influential ones.
        abs_shap = np.abs(vals.values).mean(0)
        top_2_indices = np.argsort(abs_shap)[-2:][::-1]  # Get indices of top 2 features
        for idx in top_2_indices:
            feature_name = X_subset_explain.columns[idx]
            plt.figure()
            # The 'color' parameter can show interaction effects with another feature.
            shap.plots.scatter(vals[:, feature_name], color=vals)
            plt.title(f"SHAP Dependence Plot for {feature_name} in {title}")
            plt.show()
    except Exception as e:
        print(f"Dependence plots failed: {e}")


top_idx = stats_df.index[0]
if len(stats_df) > 1:
    bot_idx = stats_df.index[-1]

    print("Explaining Top Performing Learner...")
    plot_learner_shap(top_idx, "Top Learner Importance")

    print("Explaining Bottom Performing Learner...")
    plot_learner_shap(bot_idx, "Bottom Learner Importance")
else:
    print("Only one learner in ensemble. Explaining Top Performing Learner...")
    plot_learner_shap(top_idx, "Top Learner Importance")

### 5. Local Explanation: Ensemble Confidence Breakdown
Why did the model make **this specific prediction**? We analyze a single case to see individual model 'votes' and the feature impact.

**Weighted Contribution**: This is calculated as `Base Learner Prediction * Evolved Weight`. It represents the raw influence each model had on this specific result. The red line in the bar plot represents the final aggregated ensemble probability.

In [ ]:
# Select a high-confidence positive prediction to inspect why the model is so sure
# Find a high-confidence positive prediction
all_probs = my_ensemble.predict_proba(X_test)[:, 1]
sample_idx = np.argmax(all_probs)
sample_x = X_test.iloc[[sample_idx]]

print(f"Analyzing Test Sample Index: {sample_idx}")
print(f"Ensemble Final Probability: {all_probs[sample_idx]:.4f}")
print(f"Actual Outcome: {y_test.iloc[sample_idx]}")

contributions = []
# Calculate individual model predictions and scale them by their GA-evolved weights
for model, features, weight in my_ensemble.fitted_models:
    if sample_x[features].empty:
        print(
            f"Warning: Model {type(model).__name__} has no features selected for local explanation. Assigning prediction of 0.5."
        )
        p = 0.5
    else:
        p = model.predict_proba(sample_x[features])[0, 1]
    contributions.append(
        {
            "Model": type(model).__name__,
            "Prediction": p,
            "Weight": weight,
            "Weighted_Contribution": p * weight,
        }
    )

contrib_df = pd.DataFrame(contributions)
plt.figure(figsize=(10, 6))
sns.barplot(data=contrib_df, x="Weighted_Contribution", y="Model", palette="magma")
plt.axvline(
    all_probs[sample_idx], color="red", linestyle="--", label="Final Ensemble Prob"
)
plt.title("Individual Model Contributions to a Single Prediction")
plt.legend()
plt.show()

print("\n--- Local SHAP Force Plot (Overall Ensemble) ---")
# We calculate SHAP specifically for this sample for accuracy
local_shap_values = explainer(
    sample_x, max_evals=required_max_evals_global, silent=True
)

print("1. Local Bar Plot (Feature importance for this specific prediction)")
try:
    plt.figure()
    shap.plots.bar(local_shap_values[0], max_display=10)
    plt.show()
except Exception as e:
    print(f"Local bar plot failed: {e}")

print("2. Force Plot (Directional impact)")
try:
    # Force plots are best viewed interactively in Jupyter for label handling
    shap.plots.force(local_shap_values[0])
except Exception as e:
    print(f"Could not generate force plot: {e}")

print("3. Local SHAP Waterfall Plot")
try:
    plt.figure()
    shap.plots.waterfall(local_shap_values[0], max_display=10)
    plt.show()
except Exception as e:
    print(f"Could not generate waterfall plot: {e}")

print("4. Local SHAP Decision Plot")
try:
    # Decision plots show cumulative feature impact toward the prediction
    plt.figure()
    shap.plots.decision(
        local_shap_values.base_values[0],
        local_shap_values.values[0],
        feature_names=list(X_test.columns),
    )
    plt.show()
except Exception as e:
    print(f"Could not generate decision plot: {e}")